# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset loaded:\nName: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their IDs.

We'll use the Croissant metadata structure to list all record sets and the fields (columns) inside each, printing their `@id` for reference.

In [ ]:
# List all record sets and their fields by @id

from pprint import pprint

# Gather all record set @ids from metadata (if available)
record_sets = []
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        print(f"Record Set: {rs['@id']} - {rs.get('name','')}")
        record_sets.append(rs['@id'])
        if 'field' in rs:
            print("  Fields (columns):")
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            for field in fields:
                print(f"    Field: {field['@id']} - {field.get('name', '')}")
else:
    # Alternatively, try to extract record sets from the metadata dictionary
    meta = dataset.metadata.to_json()
    if 'recordSet' in meta:
        if isinstance(meta['recordSet'], list):
            for rs in meta['recordSet']:
                rs_id = rs.get('@id', str(rs))
                print(f"Record Set: {rs_id}")
                record_sets.append(rs_id)
                # Try to get field info from inside the record set if possible
                if 'field' in rs:
                    print("  Fields (columns):")
                    for field in rs['field']:
                        print(f"    Field: {field['@id']} - {field.get('name', '')}")
        elif isinstance(meta['recordSet'], dict):
            rs = meta['recordSet']
            rs_id = rs.get('@id', str(rs))
            print(f"Record Set: {rs_id}")
            record_sets.append(rs_id)
            if 'field' in rs:
                print("  Fields (columns):")
                for field in rs['field']:
                    print(f"    Field: {field['@id']} - {field.get('name', '')}")
    else:
        print("No 'recordSet' found in metadata.")

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from available record sets (@id required)
# Here, we attempt to list them as discovered above

if not record_sets:
    # Optional fallback: try to extract record set identifiers from metadata
    meta = dataset.metadata.to_json()
    recsets = meta.get('recordSet', [])
    if isinstance(recsets, list):
        record_sets = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else str(rs) for rs in recsets]
    elif isinstance(recsets, dict):
        record_sets = [recsets.get('@id', str(recsets))]

dataframes = {}

for record_set_id in record_sets:
    records_list = list(dataset.records(record_set=record_set_id))
    if len(records_list) > 0:
        dataframes[record_set_id] = pd.DataFrame(records_list)
        print(f"Loaded {len(dataframes[record_set_id])} records for Record Set {record_set_id}")
        print("Columns:", dataframes[record_set_id].columns.tolist())
        display(dataframes[record_set_id].head())
    else:
        print(f"No records found for Record Set {record_set_id}")


## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

We'll attempt EDA on the main protein quantification record set, typically named like `protein_quantification` or similar (check printed record set ids and columns above).

In [ ]:
# Choose one of the loaded record sets for analysis
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]  # Use first available as example
    df = dataframes[main_record_set_id].copy()
    print(f"Using Record Set: {main_record_set_id}")

    # Try to guess a numeric field (e.g. 'MW', 'Coverage', 'PeptideCount', etc) from columns
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_cols:
        # Try to convert likely columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except:
                pass
        numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Analyzing numeric field: {numeric_field}")
    else:
        print("No numeric field found for analysis. Skipping EDA.")

    # Filter rows where the value > threshold
    threshold = df[numeric_field].mean() if numeric_cols else 0
    filtered_df = df[df[numeric_field] > threshold].copy() if numeric_cols else df
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field, e.g. 'Description' or any field with few unique values
    candidate_group_fields = [col for col in df.columns if df[col].nunique() < 20 and col != numeric_field]
    group_field = candidate_group_fields[0] if candidate_group_fields else None
    if group_field:
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        display(grouped_df.head())
    else:
        print("No suitable categorical group field found.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

Below, we'll plot the distribution of a numeric field and, if applicable, a boxplot by a selected group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    # Use filtered_df and numeric_field/group_field from above
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field} (> mean)")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} distribution by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion

In this notebook, we loaded the FAIR\^2 mass spectrometry dataset, explored its record set structure, extracted records by their `@id`, and applied basic filtering, normalization, and visualization workflows using field and record set `@id` references. This process can be adapted to other Croissant datasets by updating the record set and field references.

Key observations and next steps could include advanced feature engineering, analytical modeling, or biological analysis tailored to domain requirements.